### In this notebook we show how the model to model evaluation module works

In [6]:
# Standard imports
import sys
import os
import json
sys.path.append("../")
sys.path.append("../model_evaluation/")

from model_evaluation.BPMN_conversion import BPMNConverter
from model_evaluation.XML_conversion import XMLBPMNConverter

In [7]:
def load_model(path):
    """Load a BPMN model from a .json (Signavio) or .bpmn/.xml (BPMN 2.0) file.
    Returns the normalised dict ready for the evaluation pipeline.
    """
    if path.endswith(".xml") or path.endswith(".bpmn"):
        return XMLBPMNConverter.convert_file(path).to_dict()
    else:
        with open(path, "r", encoding="utf-8") as fh:
            raw = json.load(fh)
        return BPMNConverter.convert(raw).to_dict()


# Load the first model (JSON or BPMN)
path_model1 = "../examples/01 BPMN Training -T-shirt order simple.bpmn"

# Load the second model (JSON or BPMN)
path_model2 = "../examples/02 BPMN Training -T-shirt order (extended ).bpmn"


model_1_json = load_model(path_model1)
model_2_json = load_model(path_model2)

In [8]:
# ==============================================================================
# BPMN Model Comparison Pipeline
# ==============================================================================
import json
from rendering import create_similarity_dashboard, print_similarity_report
from bpmn_normalization import normalize_atomic_names
from bpmn_similarity import calculate_bpmn_similarity
from utils import cosine_sim_optimized


print("BPMN MODEL COMPARISON PIPELINE")


# Step 1: Model Summary
print("\n[1] MODEL STATISTICS")


def count_elements(model):
    """Count BPMN elements in a model."""
    return {
        "activities": len(model.get("activities", [])),
        "events": len(model.get("events", [])),
        "gateways": len(model.get("gateways", [])),
        "sequence_flows": len(model.get("sequenceFlows", [])),
        "message_flows": len(model.get("messageFlows", [])),
        "pools": len(model.get("pools", [])),
        "lanes": sum(len(p.get("lanes", [])) for p in model.get("pools", [])),
    }


model1_counts = count_elements(model_1_json)
model2_counts = count_elements(model_2_json)

print(f"Model 1: {sum(model1_counts.values())} total elements")
for key, val in model1_counts.items():
    if val > 0:
        print(f"  • {key.replace('_', ' ').title()}: {val}")

print(f"\nModel 2: {sum(model2_counts.values())} total elements")
for key, val in model2_counts.items():
    if val > 0:
        print(f"  • {key.replace('_', ' ').title()}: {val}")

# Step 2: Normalize Names
print("\n[2] SEMANTIC NAME NORMALIZATION")

threshold = 0.6
print(f"Aligning element names using a sentence transformer model (threshold={threshold})...")

model2_aligned, mappings = normalize_atomic_names(model_1_json, model_2_json, cosine_sim_optimized, threshold=threshold)

total_mappings = sum(len(v) for v in mappings.values())
if total_mappings > 0:
    print(f"✓ Applied {total_mappings} semantic name mappings")
    for elem_type, mapping in mappings.items():
        if mapping:
            print(f"  • {elem_type}: {len(mapping)} mappings")
            # Show first example
            first_old, first_new = next(iter(mapping.items()))
            print(f"    Example: '{first_old}' → '{first_new}'")
else:
    print("✓ No mappings needed (names already aligned)")

# Step 3: Calculate Similarity Without Normalization
print("\n[3] SIMILARITY ANALYSIS")


similarity_without_norm = calculate_bpmn_similarity(
    model_1_json, model_2_json, method="dice", behavioral=True
)

similarity_with_norm = calculate_bpmn_similarity(
    model_1_json, model2_aligned, method="dice", behavioral=True
)

print("WITHOUT normalization:")
print(f"  Overall Similarity: {similarity_without_norm['overall']:.1%}")
for cat in ["structural", "flows", "organizational", "subprocess", "behavioral"]:
    score = similarity_without_norm["high_level_scores"][cat]
    print(f"    • {cat.title()}: {score:.1%}")

print("\nWITH normalization:")
print(f"  Overall Similarity: {similarity_with_norm['overall']:.1%}")
for cat in ["structural", "flows", "organizational", "subprocess", "behavioral"]:
    score = similarity_with_norm["high_level_scores"][cat]
    print(f"    • {cat.title()}: {score:.1%}")

improvement = similarity_with_norm["overall"] - similarity_without_norm["overall"]
print(f"\n  → Improvement: {improvement:+.1%} ({abs(improvement)*100:.1f} percentage points)")

# Store results for dashboard
similarity_results = similarity_with_norm


print("Pipeline complete. Results stored in 'similarity_results'.")

BPMN MODEL COMPARISON PIPELINE

[1] MODEL STATISTICS
Model 1: 26 total elements
  • Activities: 7
  • Events: 2
  • Gateways: 2
  • Sequence Flows: 11
  • Pools: 1
  • Lanes: 3

Model 2: 31 total elements
  • Activities: 9
  • Events: 2
  • Gateways: 2
  • Sequence Flows: 13
  • Pools: 1
  • Lanes: 4

[2] SEMANTIC NAME NORMALIZATION
Aligning element names using a sentence transformer model (threshold=0.6)...
✓ Applied 18 semantic name mappings
  • activity_names: 7 mappings
    Example: 'Ship Goods' → 'Ship Goods'
  • activity_types: 1 mappings
    Example: 'Task' → 'Task'
  • event_names: 2 mappings
    Example: 'T-shirt order  received' → 'T-shirt order  received'
  • event_types: 2 mappings
    Example: 'StartNoneEvent' → 'StartNoneEvent'
  • gateway_names: 1 mappings
    Example: 'Custom print order?' → 'Custom print order?'
  • gateway_types: 1 mappings
    Example: 'Exclusive' → 'Exclusive'
  • pool_names: 1 mappings
    Example: 'FairTrade T-Shirt Company' → 'FairTrade T-Shirt C

In [9]:
import json
from rendering.dashboard import create_similarity_dashboard


# Create and display dashboard
dashboard = create_similarity_dashboard(
    model_1_json,
    model_2_json,
    similarity_func=cosine_sim_optimized,
    calculate_similarity_func=calculate_bpmn_similarity,
    normalize_func=normalize_atomic_names,
    initial_threshold=0.5,
)
dashboard.display()

In [5]:
# Use the reusable XML embed function
import importlib
import rendering

importlib.reload(rendering)

# Load a BPMN file and render it
with open("../examples/02 BPMN Training -T-shirt order (extended ).bpmn", "r", encoding="utf-8") as f:
    xml_str_1 = f.read()

# Navigated viewer enables zoom/pan
rendering.render_bpmn_xml_embed(xml_str_1, height_px=500, navigated=True)

# Load a BPMN file and render it
with open("../examples/01 BPMN Training -T-shirt order simple.bpmn", "r", encoding="utf-8") as f:
    xml_str_2 = f.read()

# Navigated viewer enables zoom/pan
rendering.render_bpmn_xml_embed(xml_str_2, height_px=500, navigated=True)

/Users/I585907/basevenv/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


### Trace extraction

In [6]:
from json_to_pn import parse_simplified_bpmn_json
parse_simplified_bpmn_json(model_1_json)

# parse_simplified_bpmn_json(model2_aligned)

({'sid-8F80C171-2AA4-46A5-89CA-7342B4F70267': ['sid-1CDC5406-D552-43EB-8233-DE339A0F2B7D'],
  'sid-22D7039B-5343-42CE-862F-8D7DA268E534': ['sid-288C339C-E015-4FDC-AC4C-FB05D03292EA'],
  'sid-6C12B290-4633-4A7E-B32D-DDF3E9A1E38B': ['sid-6ED911A7-38F7-44A7-BC7E-9946BCC7E3E5'],
  'sid-C8EAF1A5-2FEE-4CF8-957C-4B88847D3D50': ['sid-E29AA25D-E2EF-4049-B0E3-643263DD3320'],
  'sid-297C21FA-EEAC-48A6-9786-842D901330A0': ['sid-ABB268AA-36DA-435D-A54C-7FC7543806D6'],
  'sid-3B4FE504-F9C5-47DF-8D8D-7B9075866645': ['sid-70CDD74E-6D56-4992-9B76-226281A484D9'],
  'sid-548A488F-CAE8-4AF2-86E8-57A231D0B4A6': ['sid-DEE35855-0030-4D8E-B1F9-FCBB5243D781'],
  'sid-154A97FA-7513-45AA-9530-92C5DE6CF2C5': ['sid-CD8C9332-93E9-4ACC-9CA0-7C5005E5A0EF'],
  'sid-213C7D8B-1EE1-4E14-A3EE-5A8989A3126E': [],
  'sid-7887BD20-45AC-4E87-946C-876C83396601': ['sid-3A644C07-B720-4E43-A089-237DDD2D77B5',
   'sid-906CC8B9-E7FC-4EFD-9452-AFE77ABF8A1D'],
  'sid-3C3D4D05-4B9D-4D35-9092-E2D94D50DC8C': ['sid-6843AE8C-E2A0-4C96-A8B0

In [7]:
# Extract traces from both models
from trace_extraction import (
    extract_traces,
    compare_trace_sets,
    print_trace_comparison,
)

print("\n[4] TRACE EXTRACTION")
print("Extracting execution traces (variants) from both models...")
print("This converts models to Petri nets and explores possible execution paths.\n")

# Extract traces with activity names (more readable)
res_1 = extract_traces(
    model_1_json,
    timeout_seconds=2.0,
    max_loop_depth=3,
)

res_2 = extract_traces(
    model2_aligned,  # Use normalized version
    timeout_seconds=2.0,
    max_loop_depth=3,
)

print(f"Model 1: {len(res_1.variants)} sound variants, {len(res_1.partial_traces)} partial traces ({res_1.diagnostics.status.value})")
print(f"Model 2: {len(res_2.variants)} sound variants, {len(res_2.partial_traces)} partial traces ({res_2.diagnostics.status.value})")

if not res_1.is_sound:
    print(f"  ⚠ Model 1: {res_1.diagnostics.summary}")
if not res_2.is_sound:
    print(f"  ⚠ Model 2: {res_2.diagnostics.summary}")

# Show sample sound traces from each model
print("\nSample sound traces from Model 1 (first 3):")
for i, trace in enumerate(res_1.variants[:3], 1):
    print(f"  {i}. {' → '.join(trace)}")

print("\nSample sound traces from Model 2 (first 3):")
for i, trace in enumerate(res_2.variants[:3], 1):
    print(f"  {i}. {' → '.join(trace)}")



[4] TRACE EXTRACTION
Extracting execution traces (variants) from both models...
This converts models to Petri nets and explores possible execution paths.

Model 1: 2 sound variants, 0 partial traces (sound)
Model 2: 2 sound variants, 0 partial traces (sound)

Sample sound traces from Model 1 (first 3):
  1. T-shirt order  received → Receive Customer Order → Receive Payment → Send T-shirt to Printing Department → Print T-shirt → Receive Printed T-Shirt → Ship Goods → Receive Delivery Confirmation → Order is completed
  2. T-shirt order  received → Receive Customer Order → Receive Payment → Ship Goods → Receive Delivery Confirmation → Order is completed

Sample sound traces from Model 2 (first 3):
  1. T-shirt order  received → Receive Customer Order → Receive Payment → Send T-shirt to Printing Department → Print T-shirt → Receive Printed T-Shirt → Prepare delivery → Create and send invoice → Ship Goods → Receive Delivery Confirmation → Order is completed
  2. T-shirt order  received → 

In [8]:
# Compare trace sets comprehensively
comparison = compare_trace_sets(
    res_1,
    res_2,
    model_1_name="Model 1",
    model_2_name="Model 2 (normalized)"
)

# Print formatted comparison report (includes diagnostics for unsound nets)
print_trace_comparison(comparison, show_traces=True)


TRACE COMPARISON: Model 1 vs Model 2 (normalized)

Model 1 Statistics:
  • Variants: 2
  • Trace length: 6-9 (avg: 7.5)
  • Unique activities: 9

Model 2 (normalized) Statistics:
  • Variants: 2
  • Trace length: 8-11 (avg: 9.5)
  • Unique activities: 11

Similarity Scores:
  • Jaccard: 0.00%
  • Dice: 0.00%
  • Overlap: 0.00%

Trace Coverage:
  • Common variants: 0
  • Only in Model 1: 2
  • Only in Model 2 (normalized): 2

Unique to Model 1 (first 3):
  1. T-shirt order  received → Receive Customer Order → Receive Payment → Send T-shirt to Printing Department → Print T-shirt → Receive Printed T-Shirt → Ship Goods → Receive Delivery Confirmation → Order is completed
  2. T-shirt order  received → Receive Customer Order → Receive Payment → Ship Goods → Receive Delivery Confirmation → Order is completed

Unique to Model 2 (normalized) (first 3):
  1. T-shirt order  received → Receive Customer Order → Receive Payment → Send T-shirt to Printing Department → Print T-shirt → Receive Printe

In [9]:
# Combine structural and trace similarity for final score
structural_sim = similarity_with_norm['overall']
trace_sim = comparison['jaccard_similarity']

# Weighted combination (adjust weights as needed)
structural_weight = 0.6
trace_weight = 0.4

combined_similarity = (structural_weight * structural_sim) + (trace_weight * trace_sim)

print(f"\n{'='*70}")
print("COMBINED SIMILARITY SCORE")
print(f"{'='*70}")
print(f"Structural Similarity:  {structural_sim:.2%} (weight: {structural_weight})")
print(f"Trace Similarity:       {trace_sim:.2%} (weight: {trace_weight})")
print(f"Combined Similarity:    {combined_similarity:.2%}")
print(f"{'='*70}\n")

print("✓ Complete evaluation pipeline finished!")
print("  Models compared using both structural and behavioral (trace) similarity.")


COMBINED SIMILARITY SCORE
Structural Similarity:  91.89% (weight: 0.6)
Trace Similarity:       0.00% (weight: 0.4)
Combined Similarity:    55.13%

✓ Complete evaluation pipeline finished!
  Models compared using both structural and behavioral (trace) similarity.


In [10]:
import os

# Shows the traces for all files in examples folder

example_dir = "../examples"

trace_counts = {}
warnings_per_file = {}
errors = {}

for fname in os.listdir(example_dir):
    if not (fname.endswith(".json") or fname.endswith(".xml") or fname.endswith(".bpmn")):
        continue

    fpath = os.path.join(example_dir, fname)

    try:
        model_json = load_model(fpath)

        res = extract_traces(
            model_json,
            timeout_seconds=2.0,
            max_loop_depth=3,
        )

        trace_counts[fname] = {
            "sound": len(res.variants),
            "partial": len(res.partial_traces),
            "status": res.diagnostics.status.value,
        }
        if not res.is_sound:
            warnings_per_file[fname] = res.diagnostics.summary

    except Exception as e:
        errors[fname] = str(e)
        print(f"Unexpected error processing {fname}: {e}")


print("\nTrace counts per file:")
for k in sorted(trace_counts):
    c = trace_counts[k]
    print(f"  {k}: sound={c['sound']}, partial={c['partial']} ({c['status']})")

if warnings_per_file:
    print("\nDiagnostics for non-sound nets:")
    for k in sorted(warnings_per_file):
        print(f"  {k}: {warnings_per_file[k]}")


Trace extraction recovered partial results for net '<unnamed>': 4 sound variant(s), 2 partial trace(s); 2 distinct deadlock marking(s)
Trace extraction recovered partial results for net '<unnamed>': 21727 sound variant(s), 31963 partial trace(s); 60266 loop-cap hit(s), exploration timed out, exploration truncated by active-set cap
Trace extraction recovered partial results for net '<unnamed>': 21916 sound variant(s), 32257 partial trace(s); 60918 loop-cap hit(s), exploration timed out, exploration truncated by active-set cap
Trace extraction recovered partial results for net '<unnamed>': 0 sound variant(s), 30 partial trace(s); 1 distinct deadlock marking(s)
Trace extraction recovered partial results for net '<unnamed>': 11 sound variant(s), 5367 partial trace(s); 1 distinct deadlock marking(s), 18196 loop-cap hit(s), exploration timed out, exploration truncated by active-set cap
Trace extraction recovered partial results for net '<unnamed>': 0 sound variant(s), 4 partial trace(s); 1 d

Unexpected error processing Adrians_ex.json: Process structure error: Sequence flow sid-373935DE-B64E-4B59-A45D-B6001FF70D10 crosses subprocess boundary (from 'sid-2C01C673-6036-4212-B4BB-212F784403F0' in subprocess 'None' to 'sid-F015C8E1-D461-4980-AAD0-A2F6A7D34AFC' in subprocess 'sid-CEC28D77-4DA4-44A2-B67B-84D343033FD9').

Trace counts per file:
  01 BPMN Training -T-shirt order simple.bpmn: sound=2, partial=0 (sound)
  02 BPMN Training -T-shirt order (extended ).bpmn: sound=2, partial=0 (sound)
  03 Prepare delivery (subprocess).bpmn: sound=2, partial=0 (sound)
  E_j04.json: sound=50937, partial=68937 (unsound_recovered)
  E_j04_4.bpmn2 _ Signavio.json: sound=11, partial=5367 (unsound_recovered)
  Gateway AND.bpmn: sound=2, partial=0 (sound)
  Gateway Inclusive.bpmn: sound=2, partial=0 (sound)
  Gateway XOR.bpmn: sound=3, partial=0 (sound)
  and_gateway_no_join.bpmn: sound=0, partial=4 (unsound_no_variants)
  and_gateway_three_branches.bpmn: sound=0, partial=30 (unsound_no_variant

In [11]:
# Trace similarity for misc_booking_flight_tickets vs misc_booking_variant
# Both nets are unsound_recovered (4 sound variants + 2 partial traces each, 2 deadlock markings).
# We first normalize element names so equivalent activities align, then compare.

from bpmn_similarity import calculate_trace_similarity

booking_path = "../examples/misc_booking_flight_tickets.json"
variant_path = "../examples/misc_booking_variant.json"

booking_model = load_model(booking_path)
variant_model = load_model(variant_path)

# Step 1: Semantic name normalization (variant aligned to booking)
booking_threshold = 0.6
variant_aligned, booking_mappings = normalize_atomic_names(
    booking_model, variant_model, cosine_sim_optimized, threshold=booking_threshold
)

total_mappings = sum(len(v) for v in booking_mappings.values())
print(f"Normalization: applied {total_mappings} semantic name mappings (threshold={booking_threshold})")
for elem_type, mapping in booking_mappings.items():
    if mapping:
        print(f"  • {elem_type}: {len(mapping)} mappings")
        first_old, first_new = next(iter(mapping.items()))
        print(f"    Example: '{first_old}' → '{first_new}'")

# Step 2: Extract traces from booking and the *aligned* variant
res_booking = extract_traces(booking_model, timeout_seconds=2.0, max_loop_depth=3)
res_variant = extract_traces(variant_aligned, timeout_seconds=2.0, max_loop_depth=3)

print("\nmisc_booking_flight_tickets:",
      f"{len(res_booking.variants)} sound, {len(res_booking.partial_traces)} partial",
      f"({res_booking.diagnostics.status.value})")
print("misc_booking_variant (aligned):",
      f"{len(res_variant.variants)} sound, {len(res_variant.partial_traces)} partial",
      f"({res_variant.diagnostics.status.value})")

# Step 3: Trace similarity — sound-only and sound+partial views
sound_jaccard = calculate_trace_similarity(res_booking.variants, res_variant.variants, method="jaccard")
sound_dice    = calculate_trace_similarity(res_booking.variants, res_variant.variants, method="dice")
sound_overlap = calculate_trace_similarity(res_booking.variants, res_variant.variants, method="overlap")

merged_jaccard = calculate_trace_similarity(res_booking, res_variant, method="jaccard")
merged_dice    = calculate_trace_similarity(res_booking, res_variant, method="dice")
merged_overlap = calculate_trace_similarity(res_booking, res_variant, method="overlap")

print("\nTrace similarity — sound variants only:")
print(f"  Jaccard: {sound_jaccard:.2%}")
print(f"  Dice:    {sound_dice:.2%}")
print(f"  Overlap: {sound_overlap:.2%}")

print("\nTrace similarity — sound + partial traces:")
print(f"  Jaccard: {merged_jaccard:.2%}")
print(f"  Dice:    {merged_dice:.2%}")
print(f"  Overlap: {merged_overlap:.2%}")

# Step 4: Full comparison report (includes per-model diagnostics for unsound nets)
booking_comparison = compare_trace_sets(
    res_booking,
    res_variant,
    model_1_name="misc_booking_flight_tickets",
    model_2_name="misc_booking_variant (aligned)",
)
print_trace_comparison(booking_comparison, show_traces=True)


Trace extraction recovered partial results for net '<unnamed>': 4 sound variant(s), 2 partial trace(s); 2 distinct deadlock marking(s)
Trace extraction recovered partial results for net '<unnamed>': 4 sound variant(s), 2 partial trace(s); 2 distinct deadlock marking(s)


Normalization: applied 31 semantic name mappings (threshold=0.6)
  • activity_names: 4 mappings
    Example: 'Select best room and book' → 'Select the best offer and request tickets'
  • activity_types: 5 mappings
    Example: 'Task' → 'Task'
  • event_names: 10 mappings
    Example: 'Receive confirmation' → 'Receive confirmation'
  • event_types: 7 mappings
    Example: 'IntermediateMessageEventCatching' → 'IntermediateMessageEventCatching'
  • gateway_types: 2 mappings
    Example: 'Parallel' → 'Parallel'
  • pool_names: 3 mappings
    Example: 'Hotel Chain' → 'Travel agency'

misc_booking_flight_tickets: 4 sound, 2 partial (unsound_recovered)
misc_booking_variant (aligned): 4 sound, 2 partial (unsound_recovered)

Trace similarity — sound variants only:
  Jaccard: 33.33%
  Dice:    50.00%
  Overlap: 50.00%

Trace similarity — sound + partial traces:
  Jaccard: 50.00%
  Dice:    66.67%
  Overlap: 66.67%

TRACE COMPARISON: misc_booking_flight_tickets vs misc_booking_variant (aligned)

